# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [ ]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

## 2. Pipeline de procesamiento

In [ ]:
feature_columns = contract["features"]

# Preprocesador para arboles: imputacion por mediana (no requiere escalado)
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns),
    ]
)

# Preprocesador para modelos sensibles a escala: imputacion + StandardScaler
scaled_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            feature_columns,
        ),
    ]
)

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Escribe aquí tu respuesta.

## 3. Modelos candidatos

In [ ]:
# Pipelines completos para los tres modelos candidatos
models = {
    "DecisionTree": Pipeline([
        ("preprocessor", clone(tree_preprocessor)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "LogisticRegression": Pipeline([
        ("preprocessor", clone(scaled_preprocessor)),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "KNN": Pipeline([
        ("preprocessor", clone(scaled_preprocessor)),
        ("model", KNeighborsClassifier()),
    ]),
}

## 4. Comparación inicial con validación cruzada

In [ ]:
# Validacion cruzada estratificada de 5 folds para cada modelo
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipe in models.items():
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv, scoring="accuracy", return_train_score=True,
    )
    rows.append({
        "model": name,
        "train_accuracy": scores["train_score"].mean(),
        "val_accuracy": scores["test_score"].mean(),
        "val_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
initial_comparison

## 5. Optimización de hiperparámetros

In [ ]:
# Espacios de busqueda pequenos para cada modelo
param_grids = {
    "DecisionTree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 2, 5],
    },
    "LogisticRegression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
    },
}

# GridSearchCV con accuracy como metrica
searches = {}
rows = []
for name, pipe in models.items():
    search = GridSearchCV(
        pipe, param_grids[name],
        cv=cv, scoring="accuracy", return_train_score=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search

    best = search.best_index_
    rows.append({
        "model": name,
        "best_cv_accuracy": search.best_score_,
        "cv_std": search.cv_results_["std_test_score"][best],
        "train_accuracy": search.cv_results_["mean_train_score"][best],
        "best_params": json.dumps(search.best_params_),
    })

tuned_comparison = pd.DataFrame(rows).sort_values("best_cv_accuracy", ascending=False).reset_index(drop=True)
tuned_comparison

## 6. Selección y persistencia del campeón

In [ ]:
# Seleccion del campeon por mayor best_score_ de validacion cruzada
champion_name = max(searches, key=lambda n: searches[n].best_score_)
champion = searches[champion_name].best_estimator_
print("Campeon:", champion_name)

# Guarda el pipeline completo, los metadatos y la tabla de resultados
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "best_params": searches[champion_name].best_params_,
    "cv_accuracy": float(searches[champion_name].best_score_),
    "cv_folds": 5,
    "random_state": RANDOM_STATE,
    "n_train": int(len(X_train)),
    "features": feature_columns,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)

tuned_comparison.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

In [ ]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

**Justificación de la selección:**

> Explica la decisión considerando desempeño, variabilidad, sobreajuste, complejidad e interpretabilidad.

## Checklist

- [ ] Todo el procesamiento está dentro de pipelines.
- [ ] Comparé tres familias de modelos.
- [ ] Optimicé hiperparámetros con CV.
- [ ] Elegí el campeón sin usar test.
- [ ] Guardé el pipeline completo y sus metadatos.